# 提示词驱动抽取与别名消解（fork 增强）

一个 `prompt` 参数控制 LLM 抽取行为：目的、方法、例子、主题方向、输出要求全由提示词承载。需要 `DEEPSEEK_API_KEY`（或其他 provider 的 key）——没有 key 时本 notebook 的 LLM 单元自动跳过。文档见「提示词驱动的抽取」（docs_zh/guides/prompt-driven-extraction.md）。

In [ ]:
import os

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
RUN_LLM = bool(DEEPSEEK_API_KEY)
print("LLM cells:", "RUN" if RUN_LLM else "SKIP（无 key）")

## 1. 抽取前：别名消解

先把「岭川」「华跃北京」这类简称归并成规范全称，避免抽取结果碎片化。`resolve_aliases_llm` 带子串保护——「北京」不会误替换进「华跃控股北京有限公司」：

In [ ]:
text = (
    "岭川与华跃控股北京有限公司签订2025年商业政策。\n"
    "华跃北京负责北京地区分销；政策编号 DEMO-POLICY-2025-026，"
    "有效期2025年1月1日至2025年12月31日。\n"
    "每季度结算一次销售折扣。"
)

if RUN_LLM:
    from semantica.semantic_extract import resolve_aliases_llm
    result = resolve_aliases_llm(
        text, provider="deepseek", llm_model="deepseek-flash",
        api_key=DEEPSEEK_API_KEY, min_confidence=0.85,
    )
    resolved_text = result["resolved_text"]
    print("mappings:", result["mappings"])
    print("skipped:", result["skipped"])
else:
    resolved_text = text.replace("岭川", "杭州岭川制药有限公司")
    resolved_text = resolved_text.replace("华跃北京", "华跃控股北京有限公司")
print(resolved_text[:60])

## 2. 提示词驱动的实体抽取

`prompt` 整体替换默认指令；框架自动附加 schema 守卫尾与原文注入：

In [ ]:
PROMPT = '''你是医药商业政策解析器。只抽取：
- 公司全称（ORG）
- 政策编号与政策名称（POLICY）
- 日期区间（DATE）

例子：「华跃控股北京有限公司（以下简称"华跃北京"）」→ 抽「华跃控股北京有限公司」（ORG）

输出要求：宁缺毋滥，confidence 低于 0.7 的不要。'''

if RUN_LLM:
    from semantica.semantic_extract import NERExtractor
    ner = NERExtractor(method="llm_prompt", provider="deepseek", llm_model="deepseek-flash",
                       api_key=DEEPSEEK_API_KEY, prompt=PROMPT)
    entities = ner.extract(resolved_text)
    for e in entities:
        print(f"  {e.text!r} [{e.label}] conf={e.confidence:.2f}")
else:
    entities = []
    print("（无 key，跳过）")

## 3. 关系与三元组（同一提示词风格）

换一个针对关系的 prompt，链路完全一致；EventDetector 经 NER/Relation 组合自动受益：

In [ ]:
REL_PROMPT = '''只抽取签约/分销/结算类商务关系。谓词用英文蛇形
（signs_with / distributes_for / settles_quarterly）。
输出要求：只保留有原文依据的关系。'''

if RUN_LLM:
    from semantica.semantic_extract import RelationExtractor, TripletExtractor
    rel = RelationExtractor(method="llm_prompt", provider="deepseek", llm_model="deepseek-flash",
                            api_key=DEEPSEEK_API_KEY, prompt=REL_PROMPT)
    relations = rel.extract_relations(resolved_text, entities)
    trip = TripletExtractor(method="llm_prompt", provider="deepseek", llm_model="deepseek-flash",
                            api_key=DEEPSEEK_API_KEY, prompt=REL_PROMPT)
    triplets = trip.extract(resolved_text, entities=entities, relations=relations)
    for r in relations:
        print(f"  {r.subject} --[{r.predicate}]--> {r.object}")
else:
    relations, triplets = [], []

## 4. 建图

规范化 → 消解 → 抽取的产物直接进 `GraphBuilder`：

In [ ]:
from semantica.kg import GraphBuilder

graph = GraphBuilder(merge_entities=True).build(
    {"entities": entities, "relationships": relations})
print(f"Graph: {len(graph['entities'])} nodes, {len(graph['relationships'])} edges")

## 要点回顾

- `prompt=None`（缺省）＝官方默认行为，零变化
- prompt 进缓存键：同文本不同指令不串结果
- 别名消解的 `min_confidence`（默认 0.85）防幻觉映射
- 文档：docs_zh/guides/prompt-driven-extraction.md